# M1: Sklearn — Logistic Regression on Titanic

Load the Titanic dataset, wrap a `LogisticRegression` in `SklearnModel`,
use stub `Loss`/`Optimizer`, and override `train()` to call `.fit()`.

**Prerequisites:** scikit-learn, pandas (both lazy-imported by adapters)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder

from pipeline.config import Config
from pipeline.data.csv_source import CsvDataSource
from pipeline.data.split import train_test_split
from pipeline.data.utils import collect_arrays
from pipeline.evaluation.metrics import Metrics, accuracy, f1_score, confusion_matrix
from pipeline.pipeline import BasePipeline, PipelineState
from pipeline.adapters.sklearn_adapter import SklearnModel, StubLoss, StubOptimizer

print('Imports OK')

## 1. Fetch Titanic from OpenML

In [ ]:
# Fetch real Titanic data (no Kaggle download needed)
titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
df = titanic.data.copy()
df['survived'] = titanic.target.astype(int)

# Simple preprocessing
cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = df[cols].copy()
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df['embarked'] = LabelEncoder().fit_transform(df['embarked'].astype(str))
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['embarked'] = df['embarked'].fillna(0)
df = df.astype(float)
df['survived'] = df['survived'].astype(int)

print(f'Data shape: {df.shape}')
print(f'Survival rate: {df.survived.mean():.2%}')
df.head()

In [ ]:
# Scale features
feature_cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X = df[feature_cols].values.astype(np.float64)
y = df['survived'].values.astype(np.int64)
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

# Write to temp CSV (pipeline reads from file)
import tempfile, os
_tmpdir = tempfile.mkdtemp()
csv_path = os.path.join(_tmpdir, 'titanic.csv')
scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
scaled_df['survived'] = y
scaled_df.to_csv(csv_path, index=False)
print(f'Wrote {csv_path}')

## 2. Define the Sklearn Pipeline

In [ ]:
config = Config(
    batch_size=64, num_epochs=1, train_ratio=0.8, seed=42,
    output_dir=f'{_tmpdir}/output', task_name='titanic-sklearn',
)


class TitanicSklearnPipeline(BasePipeline):
    def load_data(self, state: PipelineState) -> None:
        source = CsvDataSource(
            csv_path, batch_size=config.batch_size,
            target_column='survived', shuffle=True, seed=config.seed,
        )
        train, val = train_test_split(source, train_ratio=config.train_ratio)
        state.data_stream = train
        state.val_data_stream = val

    def build_model(self, state: PipelineState) -> None:
        state.model = SklearnModel(
            LogisticRegression(max_iter=1000, random_state=config.seed)
        )
        state.loss_fn = StubLoss()
        state.optimizer = StubOptimizer()
        state.metrics = Metrics(accuracy=accuracy, f1=f1_score)

    def train(self, state: PipelineState) -> None:
        X, y = collect_arrays(state.data_stream)
        state.model._estimator.fit(X, y)
        state.history = {'loss': [0.0]}

    def evaluate(self, state: PipelineState) -> None:
        state.model.eval_mode()
        preds, trues = [], []
        for batch in state.val_data_stream:
            probs = np.asarray(state.model.forward(batch.inputs))
            preds.append(np.argmax(probs, axis=1))
            trues.append(np.asarray(batch.targets))
        y_pred = np.concatenate(preds)
        y_true = np.concatenate(trues).astype(np.int64)
        state.metrics.compute(y_true, y_pred)

    def export(self, state: PipelineState) -> None:
        state.predictions = np.array([0])

## 3. Run

In [ ]:
state = TitanicSklearnPipeline(config).run('train')
print(f'Accuracy: {state.metrics["accuracy"]:.4f}')
print(f'F1:       {state.metrics["f1"]:.4f}')

## What happened?

1. `SklearnModel` wrapped a `LogisticRegression` — `forward()` delegates to `predict_proba()`
2. `StubLoss` and `StubOptimizer` satisfy the pipeline contract — they do nothing,
   because sklearn's `.fit()` handles training internally
3. The pipeline overrode `train()` to call `_estimator.fit(X, y)` on the full dataset,
   instead of the per-batch gradient descent loop
4. `evaluate()` used `predict_proba()` → `argmax` to compute accuracy and F1

**Key insight:** sklearn and gradient descent train differently. The pipeline
accommodates both — sklearn overrides `train()`, numpy uses the default `TrainLoop`.